<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">یک شبکهٔ کوچک واقعاً چه چیزی یاد می‌گیرد؟</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چرا <bdi dir="ltr">Activation</bdi> می‌تواند نتیجهٔ یادگیری <bdi dir="ltr">XOR</bdi> را تغییر دهد؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/18-module.html"><bdi dir="ltr">18-module</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/19-network.html"><bdi dir="ltr">19-network</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">چهار ورودی <bdi dir="ltr">XOR</bdi> را خودمان می‌سازیم؛ نمونهٔ <bdi dir="ltr">Test</bdi> مستقلی نداریم. موفقیت روی این چهار نقطه فقط یادگیری همین جدول را نشان می‌دهد. پیش از اجرا تعداد <bdi dir="ltr">Parameter</bdi>های شبکهٔ ۲→۸→۲ با <bdi dir="ltr">Bias</bdi> را حساب کنید.</p>
</div>

In [ ]:
from torch import nn
from torch.nn import functional as F
x = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
targets = torch.tensor([0,1,1,0], dtype=torch.long)
inspect("features", x)
inspect("class IDs", targets)

def train_network(nonlinear, width=8, seed=9):
    torch.manual_seed(seed)
    network = nn.Sequential(nn.Linear(2,width),
                            nn.Tanh() if nonlinear else nn.Identity(),
                            nn.Linear(width,2))
    optimizer = torch.optim.Adam(network.parameters(), lr=0.05)
    history = []
    for _ in range(400):
        optimizer.zero_grad(set_to_none=True)
        logits = network(x)
        loss = F.cross_entropy(logits, targets)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return network, history

network, history = train_network(True)
assert sum(p.numel() for p in network.parameters()) == 42
inspect("logits", network(x))
print("Predictions:", network(x).argmax(-1).tolist())
assert network(x).argmax(-1).tolist() == targets.tolist()


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط <bdi dir="ltr">Activation</bdi> را حذف کنیم</h2><p style="text-align:right">شبکه و <bdi dir="ltr">Optimizer</bdi> را از نو می‌سازیم و همان <bdi dir="ltr">Seed</bdi> را می‌گذاریم. <code dir="ltr" style="unicode-bidi:isolate;direction:ltr;text-align:left">nn.Identity()</code> ورودی را بی‌تغییر عبور می‌دهد و اینجا جای <bdi dir="ltr">Activation</bdi> را می‌گیرد. دو تبدیل <bdi dir="ltr">Affine</bdi> بدون <bdi dir="ltr">Activation</bdi> در یک تبدیل <bdi dir="ltr">Affine</bdi> خلاصه می‌شوند. پیش‌بینی کنید چرا جداسازی <bdi dir="ltr">XOR</bdi> برای این مسیر ممکن نیست.</p>
</div>

In [ ]:
linear, linear_history = train_network(False)
print("Without activation:", linear(x).argmax(-1).tolist())
fig, ax = plt.subplots(figsize=(6,3))
ax.plot(history, label="Tanh")
ax.plot(linear_history, label="No activation")
ax.set(xlabel="Step", ylabel="Training Loss")
ax.legend()
plt.show()
try:
    F.cross_entropy(network(x), targets.float())
except RuntimeError as error:
    print("Expected class-ID dtype failure:", error)
else:
    raise AssertionError("Integer class IDs are required here")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> اجرای مرجع بالا را نگه دارید. در <bdi dir="ltr">Cell</bdi> تازه، تابع <code dir="ltr" style="unicode-bidi:isolate;direction:ltr;text-align:left">train_network(True, width=2, seed=4)</code> را فراخوانی کنید و مدل و تاریخچهٔ تازه را بگیرید؛ سپس تعداد <bdi dir="ltr">Parameter</bdi> و دقت آن را گزارش کنید، بی‌آنکه موفقیت کامل را <bdi dir="ltr">assert</bdi> کنید. چند <bdi dir="ltr">Seed</bdi> را جدا بسنجید. موفقیت یا شکست یک اجرای کوتاه را با اثبات ظرفیت یا تعمیم یکی نگیرید. در مدل زبان، همین تفکیکِ <bdi dir="ltr">Forward</bdi>، <bdi dir="ltr">Loss</bdi>، <bdi dir="ltr">Gradient</bdi> و <bdi dir="ltr">Step</bdi> را با داده‌ای بزرگ‌تر دنبال خواهیم کرد.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/19-network.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: دو <bdi dir="ltr">Linear</bdi> بدون <bdi dir="ltr">Activation</bdi> را در یکی خلاصه کنید</h2>
<p style="text-align:right">با وزن و <bdi dir="ltr">Bias</bdi> واقعی نشان دهید زیادکردن <bdi dir="ltr">Layer</bdi>‌های خطی به‌تنهایی غیرخطی‌بودن نمی‌سازد. پیش‌نیاز: نمونهٔ <bdi dir="ltr">XOR</bdi> و تفاوت <bdi dir="ltr">Tanh</bdi> با <bdi dir="ltr">Identity</bdi> در همین دفتر را دیده‌اید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">first</code> از ۲ به ۴ ویژگی و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">last</code> از ۴ به ۳ برود، وزن تبدیل ترکیبی چه شکل دارد؟ <bdi dir="ltr">Bias</bdi> اول پس از عبور از وزن دوم چه می‌شود؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch import nn
first,last = nn.Linear(2,4),nn.Linear(4,3)
x = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
print('two-layer output:',last(first(x)).detach())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">collapse_linear(first,last)</code> زوج <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(weight,bias)</code> یک <bdi dir="ltr">Linear</bdi> هم‌ارز را برگرداند. میان این دو <bdi dir="ltr">Layer Activation</bdi> وجود ندارد؛ وزن‌های ورودی را تغییر ندهید.</p>
</div>

In [ ]:
def collapse_linear(first, last):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = collapse_linear(first,last)
    if result is None: return False
    w,b = result
    assert w.shape == (3,2) and b.shape == (3,)
    torch.testing.assert_close(x@w.T+b,last(first(x)))
    a,c = nn.Linear(3,5),nn.Linear(5,2)
    z = torch.randn(2,4,3)
    w,b = collapse_linear(a,c)
    torch.testing.assert_close(z@w.T+b,c(a(z)))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <bdi dir="ltr">Tanh</bdi> را بین همان دو <bdi dir="ltr">Layer</bdi> اضافه کنید؛ وزن‌ها ثابت بمانند. تبدیل ترکیبی قبلی دیگر همان مسیر نیست.</p>
</div>

In [ ]:
with torch.no_grad():
    affine = last(first(x))
    nonlinear = last(torch.tanh(first(x)))
print('same weights, activation difference:',nonlinear-affine)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب <bdi dir="ltr">Bias</bdi> اول را حذف می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">composed_bias(first,last)</code> فقط <bdi dir="ltr">Bias</bdi> صحیحِ ترکیب را برگرداند.</p>
</div>

In [ ]:
with torch.no_grad():
    first.bias.fill_(1.)
wrong_bias = last.bias
print('lost contribution:',(last.weight@first.bias).detach())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def composed_bias(first, last):
    # TODO
    return None

In [ ]:
def test_repair():
    result = composed_bias(first,last)
    if result is None: return False
    torch.testing.assert_close(result,last(first(torch.zeros(2))))
    a,b = nn.Linear(1,2),nn.Linear(2,1)
    torch.testing.assert_close(composed_bias(a,b),b(a(torch.zeros(1))))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><bdi dir="ltr">FFN</bdi> در <bdi dir="ltr">Transformer</bdi> نیز دو <bdi dir="ltr">Linear</bdi> دارد، اما <bdi dir="ltr">GELU</bdi> میان آن‌ها اجازه نمی‌دهد کل مسیر را به این شیوه در یک <bdi dir="ltr">Linear</bdi> ثابت خلاصه کنیم.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا افزایش تعداد <bdi dir="ltr">Linear</bdi>ها، بدون <bdi dir="ltr">Activation</bdi>، پاسخ محدودیت شبکهٔ خطی روی <bdi dir="ltr">XOR</bdi> نیست؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/19-network.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-05_first_network.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>